<a href="https://colab.research.google.com/github/PerryGu/audio_mood_classifier_hf/blob/main/audio_mood_classifier_hf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Project Setup, Cloning & Data Preparation
This step authenticates with GitHub using your personal token and clones the latest repository version. It then mounts your Google Drive, locates the zipped MP3 dataset, and automatically handles extraction to the project structure (detecting the folder format and skipping extraction if the processed dataset already exists).

In [2]:
from google.colab import userdata, drive
import zipfile, os
from pathlib import Path

# ── 1. Git clone ──────────────────────────────────────────────────────────────
print("=" * 55)
print("STEP 1 — Cloning repository from GitHub...")
print("=" * 55)

git_token = userdata.get('GIT_TOKEN')
repo_url = f"https://{git_token}@github.com/PerryGu/audio_mood_classifier_hf.git"
!rm -rf audio_mood_classifier_hf
!git clone {repo_url}
%cd audio_mood_classifier_hf
print("✓ Repository ready.\n")

# ── 2. Mount Google Drive ─────────────────────────────────────────────────────
print("=" * 55)
print("STEP 2 — Mounting Google Drive...")
print("=" * 55)
drive.mount("/content/drive")
print("✓ Drive mounted.\n")

# ── 3. Data setup ─────────────────────────────────────────────────────────────
print("=" * 55)
print("STEP 3 — Checking data...")
print("=" * 55)

ZIP_PATH   = "/content/drive/MyDrive/audio_mood_classifier_hf/data/mp3_data.zip"
EXTRACT_TO = "data/mp3_data"

if os.path.exists("data/processed_dataset") and os.listdir("data/processed_dataset"):
    print("✓ processed_dataset already in repo — no MP3 extraction needed.")
    print("  The pipeline will load directly from the cache.\n")
else:
    print("  No processed_dataset found — will extract MP3s from Drive zip.")
    print(f"  Zip path : {ZIP_PATH}")
    print(f"  Target   : {EXTRACT_TO}\n")

    if not os.path.exists(ZIP_PATH):
        print(f"✗ ERROR: zip not found at '{ZIP_PATH}'")
        print("  Check the ZIP_PATH variable and make sure Drive is mounted correctly.")
    else:
        zip_size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
        print(f"  Found zip ({zip_size_mb:.1f} MB). Extracting — this may take a minute...")

        os.makedirs("data", exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            total = len(zf.namelist())
            print(f"  {total} entries in zip. Detecting structure...")
            top_level = {n.split("/")[0] for n in zf.namelist() if n.strip("/")}
            print(f"  Top-level folders: {sorted(top_level)}")

            if len(top_level) == 1:
                print(f"  Single wrapper folder detected — extracting to 'data/' ...")
                zf.extractall("data")
                extracted = os.path.join("data", next(iter(top_level)))
                if extracted != EXTRACT_TO:
                    print(f"  Renaming '{extracted}' → '{EXTRACT_TO}' ...")
                    os.rename(extracted, EXTRACT_TO)
            else:
                print(f"  Category folders at root — extracting directly into '{EXTRACT_TO}' ...")
                os.makedirs(EXTRACT_TO, exist_ok=True)
                zf.extractall(EXTRACT_TO)

        mp3_count = len(list(Path(EXTRACT_TO).rglob("*.mp3")))
        print(f"\n✓ Extraction complete — {mp3_count} MP3 files ready in '{EXTRACT_TO}'.\n")


# ── Ready ─────────────────────────────────────────────────────────────────────
print("=" * 55)
print("Setup complete. Run the next cell to start training.")
print("=" * 55)

STEP 1 — Cloning repository from GitHub...
Cloning into 'audio_mood_classifier_hf'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 141 (delta 78), reused 104 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 113.19 KiB | 3.23 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/audio_mood_classifier_hf
✓ Repository ready.

STEP 2 — Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Drive mounted.

STEP 3 — Checking data...
  No processed_dataset found — will extract MP3s from Drive zip.
  Zip path : /content/drive/MyDrive/audio_mood_classifier_hf/data/mp3_data.zip
  Target   : data/mp3_data

  Found zip (248.8 MB). Extracting — this may take a minute...
  1420 entries in zip. Detecting structure...
  Top-level folders: ['mp3_data']
  Single wrapper fol

### 2. Update and Sync (In case of a new update to the code)
Pulls the latest code changes (git commits) from the main branch to ensure your local Colab runtime is up-to-date with the latest refactorings and bug fixes.

In [5]:
!git pull origin main

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.58 KiB | 404.00 KiB/s, done.
From https://github.com/PerryGu/audio_mood_classifier_hf
 * branch            main       -> FETCH_HEAD
   991d04a..ea73855  main       -> origin/main
Updating 991d04a..ea73855
Fast-forward
 README.md               | 18 +++++++++++++-----
 main.py                 | 19 +++++++++----------
 src/pipeline_manager.py | 25 +++++++++++++++++--------
 3 files changed, 39 insertions(+), 23 deletions(-)


### Checks the type of GPU currently available in the session. If you receive command not found, it means that there is no GPU currently available.

In [6]:
!nvidia-smi

Thu Jul 16 14:53:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 3. Execution
Initializes the project environment and executes the main pipeline script. This assumes your data is already accessible or will be processed according to the defined logic.

In [7]:
import os
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = token
login(token=token, add_to_git_credential=False)
print("Authenticated ✓")

!python main.py

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Authenticated ✓
[INFO] Colab: working directory set to '/content/audio_mood_classifier_hf'.
[INFO] Pipeline Manager initialized.
Starting the main function
[INFO] Execution Mode : BOTH
[INFO] Output folder  : /content/audio_mood_classifier_hf/models/mood_classifier_2026-07-16_14-54
[INFO] Session name   : mood_classifier_2026-07-16_14-54_14-54
[INFO] TensorBoard    : /content/audio_mood_classifier_hf/runs/mood_classifier_2026-07-16_14-54_14-54
[INFO] Continuous log : /content/audio_mood_classifier_hf/runs/continuous/mood_classifier_2026-07-16_09-27
[INFO] Resuming weights from: /content/audio_mood_classifier_hf/models/mood_classifier_2026-07-16_09-27/checkpoint-372
[INFO] LR: 4e-07  |  Batch: 32  |  Epochs: 10

[INFO] Colab: installing missing runtime packages...
[INFO] Colab: packages ready.
[INFO] Google Colab environment detected.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
[INFO] Hugging Face Hub: 